In [1]:
import os
from dotenv import load_dotenv

# scaper
from tavily import TavilyClient

# output class
from pydantic import BaseModel
from typing import Optional
from datetime import datetime

# langchain
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [2]:
load_dotenv()

True

In [3]:
tool = TavilySearchResults(
    max_results=5,
    search_depth="basic" # or advanced
)

llm = ChatOpenAI(model="gpt-4o-mini")

In [4]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """Jesteś agentem który pobiera newsy z internetu.
    Dla każdego artykułu zwróć dane w formacie:
    - title: tytuł
    - url: link
    - summary: krótkie streszczenie (2-3 zdania)
    - topic: kategoria (AI / Tech / Finance / Politics / Other)
    - source: nazwa serwisu
    """),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

In [5]:
# agent
agent = create_tool_calling_agent(llm, [tool], prompt)
executor = AgentExecutor(agent=agent, tools=[tool], verbose=True)

In [6]:
result = executor.invoke({
    "input": "Znajdź 5 najnowszych newsów o AI z dzisiaj"
})



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'najnowsze newsy AI'}`


[{'title': 'AI News - hAI Magazine', 'url': 'https://haimagazine.com/pl/kategoria/ai_news/', 'content': 'Deep Research już na Gemini 3 Pro. Google prezentuje nową wersję agenta badawczego Deep Research, opartą na Gemini 3 Pro. Narzędzie ma automatyzować złożone zadania analityczne, a premiera w dniu ogłoszenia GPT-5.2 podkreśla nasilającą się rywalizację z OpenAI. Najnowsza analiza danych z OpenRouter pokazuje, że sztuczna inteligencja jest w znacznie mniejszym stopniu wykorzystywana do pisania maili czy podsumowywania dokumentów, a w dużo większym – do rozmowy i kreatywnego odgrywania ról. AI w firmach ma się zwracać. Zamiast ścigać się na parametry modeli Amazon pokazał ekosystem agentów i narzędzi, które mają sprawić, że sztuczna inteligencja zacznie wykonywać realną pracę, a nie tylko produkować odpowiedzi. Gemini 3 w jednej chwili pokazał, że przewaga rynkowa ni

In [7]:
print(result["output"])

Oto pięć najnowszych wiadomości dotyczących sztucznej inteligencji:

1. **title:** [Google's on a roll. Deep Research is now on Gemini 3 Pro](https://haimagazine.com/en/ai-news-2/googles-on-a-roll-deep-research-is-now-on-gemini-3-pro/)
   - **url:** https://haimagazine.com/en/ai-news-2/googles-on-a-roll-deep-research-is-now-on-gemini-3-pro/
   - **summary:** Google zaprezentował nową wersję narzędzia Deep Research, opartego na architekturze Gemini 3 Pro, które ma za zadanie automatyzować złożone zadania analityczne. Premiera zbiega się z zapowiedzią aktualizacji GPT-5.2, co podkreśla rosnącą rywalizację pomiędzy Google a OpenAI. 
   - **topic:** AI
   - **source:** hAI Magazine

2. **title:** [Elon Musk Unveils Grok 3, Says It Outperforms All AI Rivals](https://finance.yahoo.com/news/elon-musk-unveils-grok-3-152425872.html)
   - **url:** https://finance.yahoo.com/news/elon-musk-unveils-grok-3-152425872.html
   - **summary:** Elon Musk zaprezentował Grok 3, nowego chatbota AI, który wed